# Test Availability Cases

In [ ]:
from pathlib import Path

from carwatch.io import load_saliva, load_study_manager_export
from carwatch.logs import extract_awakening_events_from_raw_logs, extract_sample_events_from_raw_logs, extract_day_summary_from_summary, extract_awakening_events_from_summary, extract_sample_events_from_summary
from carwatch.merge import merge_saliva

%load_ext autoreload
%autoreload 2

In [ ]:
DATA_DIR = Path("../example_data")
if not DATA_DIR.exists():
    DATA_DIR = Path("data")
DATA_DIR

## Study Manager export

`load_study_manager_export` retains one row per participant. Its column levels identify the day, sample, and variable without repeating day-level information for every sample.

In [ ]:
study_results = load_study_manager_export(
    DATA_DIR / "study_manager_summary.csv",
    tz="Europe/Berlin",
)
study_results

## Focused Study Manager views

The helper functions extract day-level awakening information and sample-level observations only when those representations are needed. The original mismatch summary remains available once per day.

In [ ]:
study_awakening = extract_awakening_events_from_summary(study_results)
study_days = extract_day_summary_from_summary(study_results)
study_samples = extract_sample_events_from_summary(study_results)
study_awakening

In [ ]:
study_days

In [ ]:
study_samples

## Audit recorded sample swaps

The per-sample mismatch flag is derived from `scheduled_sample` and `recorded_sample`.

In [ ]:
mismatches = study_samples.loc[study_samples["sample_mismatch"].fillna(False)]
mismatches

## Merge saliva measurements

The merge uses `recorded_sample` from the app. Swapped tubes are assigned to the scheduled sampling position at which they were actually collected.

In [ ]:
saliva = load_saliva(DATA_DIR / "saliva_samples.csv")
merged_samples = merge_saliva(study_results, saliva, missing_carwatch_data="ignore")
merged_samples

In [ ]:
from carwatch.exceptions import MergeError

try:
    merge_saliva(study_results, saliva, missing_carwatch_data="raise")
except MergeError as error:
    print(error)

## Impute missing CARWatch sampling times

The theoretical schedule is given in minutes relative to awakening. Only B3 has a laboratory value but no CARWatch event, so only its sampling time is imputed.

In [ ]:
imputed_samples = merge_saliva(
    study_results,
    saliva,
    missing_carwatch_data="impute",
    sampling_schedule=[15, 30, 45, 60],
)
imputed_samples